In [1]:
!pip -q install pinecone sentence-transformers langchain langchain-google-genai pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 11.2 MB/s eta 0:00:00


In [2]:
import os
import time
import yaml
import numpy as np

from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [3]:
from google.colab import userdata
openai_key = userdata.get('OPENAI_API_KEY')
google_key = userdata.get('GOOGLE_API_KEY')
pinecone_api_key = userdata.get('PINECONE_API_KEY')
# os.environ["GOOGLE_API_KEY"] = google_key

### Step 1 - Get your LLM ready

In [4]:
llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash', api_key=google_key)
print(llm.__class__.__name__)

ChatGoogleGenerativeAI


### Step 2 - Access your Pinecone Index and Embedding model

In [5]:
pc = Pinecone(api_key=pinecone_api_key)

index_name = "vector-docs"
index = pc.Index(index_name)

model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Step 3 - Editing the query (customize)
 - Multi Query Expansion
 - HyDE

In [6]:
from collections import defaultdict
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
def generate_multi_queries(query: str, n: int = 3):
    prompt = f"""
You are helping with retrieval for a financial chatbot.

Generate {n} short alternative search queries for the user query below.
Keep them close in meaning.
Make them useful for document retrieval.
Return only the queries, one per line.
Do not number them.

User query: {query}
""".strip()

    text = llm.invoke(prompt).content.strip()
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    queries = [query] + lines
    queries = list(dict.fromkeys(queries))
    return queries

def make_keywords(text, max_words=6):
    words = (
        text.lower()
        .replace("?", "")
        .replace(",", "")
        .replace(".", "")
        .replace("(", "")
        .replace(")", "")
        .split()
    )
    words = list(dict.fromkeys(words))
    return words[:max_words]


In [9]:
#queries = generate_multi_queries("How do I get a home loan?", n = 5)
#print(queries)

In [10]:
queries = ['How do I get a home loan?',
 'Mortgage application process',
 'Requirements for a home loan',
 'Best home loan rates today',
 'First-time home buyer programs']

### Step 4 - Retrieval

In [11]:
# def retrieve_docs(query_text, keywords=None, top_k=5):
#     query_vector = model.encode([query_text])[0].tolist()

#     if keywords:
#         results = index.query(
#             vector=query_vector,
#             top_k=top_k,
#             include_metadata=True,
#             filter={"keywords": {"$in": keywords}}
#         )
#     else:
#         results = index.query(
#             vector=query_vector,
#             top_k=top_k,
#             include_metadata=True
#         )

#     response = []
#     for match in results["matches"]:
#         response.append({
#             "id": match["id"],
#             "score": float(match["score"]),
#             "text": match["metadata"]["text"]
#             # "links":
#         })
# # Home work - Put links from metadata in the response
#     return response

In [12]:
def retrieve_docs_single(query_text, top_k=5):
    keywords = make_keywords(query_text, max_words=6)
    query_vector = model.encode([query_text])[0].tolist()

    if keywords:
        results = index.query(
            vector=query_vector,
            top_k=top_k,
            include_metadata=True,
            filter={"keywords": {"$in": keywords}}
        )
    else:
        results = index.query(
            vector=query_vector,
            top_k=top_k,
            include_metadata=True
        )

    response = []
    for rank, match in enumerate(results["matches"], start=1):
        response.append({
            "id": match["id"],
            "score": float(match["score"]),
            "rank": rank,
            "text": match["metadata"]["text"]
        })

    return response

def fuse_results_rrf(list_of_ranked_lists, final_top_k=5, k=60):
    """
    Reciprocal Rank Fusion (RRF)
    score(doc) += 1 / (k + rank)
    """
    doc_store = {}
    rrf_scores = defaultdict(float)

    for ranked_list in list_of_ranked_lists:
        for item in ranked_list:
            doc_id = item["id"]
            rank = item["rank"]

            doc_store[doc_id] = item
            rrf_scores[doc_id] += 1 / (k + rank)

    fused = []
    for doc_id, item in doc_store.items():
        fused.append({
            "id": item["id"],
            "score": rrf_scores[doc_id],   # fused score now
            "text": item["text"]
        })

    fused = sorted(fused, key=lambda x: x["score"], reverse=True)
    return fused[:final_top_k]

def retrieve_docs_multi(queries, top_k_per_query=4, final_top_k=5):
    all_ranked_lists = []

    for q in queries:
        docs = retrieve_docs_single(q, top_k=top_k_per_query)
        all_ranked_lists.append(docs)

    fused_results = fuse_results_rrf(
        all_ranked_lists,
        final_top_k=final_top_k,
        k=60
    )
    return fused_results

In [13]:
retrieve_docs_single(queries[0])

[{'id': 'home_loan.txt_chunk_0',
  'score': 0.597376525,
  'rank': 1,
  'text': 'BrightBridge Finance — Home Loan\nSimple loans. Clear terms. Fast decisions.\n\nOverview\nA Home Loan is a secured loan used for purchasing or constructing a residential property, and in some programs, for renovation or balance transfer. Because the loan is backed by the property, home loans generally offer longer tenors and may offer more favorable pricing than unsecured credit, subject to your profile and property evaluation.\n\nWhat you can finance\n• Purchase of a ready-to-move-in home\n• Purchase of a resale property\n• Construction on an owned plot (subject to approvals)\n• Home improvement/renovation (select programs)\n• Balance transfer (moving an existing home loan from another lender), subject to evaluation'},
 {'id': 'FAQs.txt_chunk_6',
  'score': 0.533958495,
  'rank': 2,
  'text': '5) Product-Specific FAQs\n\nPersonal Loan\nQ18. What can I use a personal loan for?\nA. Common uses include medic

In [ ]:
retrieve_docs_multi(queries)

[{'id': 'home_loan.txt_chunk_0',
  'score': 0.06530936012691699,
  'text': 'BrightBridge Finance — Home Loan\nSimple loans. Clear terms. Fast decisions.\n\nOverview\nA Home Loan is a secured loan used for purchasing or constructing a residential property, and in some programs, for renovation or balance transfer. Because the loan is backed by the property, home loans generally offer longer tenors and may offer more favorable pricing than unsecured credit, subject to your profile and property evaluation.\n\nWhat you can finance\n• Purchase of a ready-to-move-in home\n• Purchase of a resale property\n• Construction on an owned plot (subject to approvals)\n• Home improvement/renovation (select programs)\n• Balance transfer (moving an existing home loan from another lender), subject to evaluation'},
 {'id': 'FAQs.txt_chunk_6',
  'score': 0.06478053939714437,
  'text': '5) Product-Specific FAQs\n\nPersonal Loan\nQ18. What can I use a personal loan for?\nA. Common uses include medical expense

### Step 5 - Context building (customizations)
- Concatenate the context together
- Post filtering
- Add links
- shorten the context
- Summarize


In [14]:
def build_context(results, max_chars: int = 1200) -> str:
    parts = []
    total_chars = 0

    for item in results:
        chunk = item["text"].strip()

        remaining = max_chars - total_chars
        if remaining <= 0:
            break

        if len(chunk) > remaining:
            chunk = chunk[:remaining]

        parts.append(chunk)
        total_chars += len(chunk)

    return "\n\n".join(parts)

### Step 6 - Prompt Rendering and Augmentation (customizations)

In [15]:
session_store = {}
def get_session_history(session_id: str):
  if(session_id not in session_store):
    session_store[session_id] = InMemoryChatMessageHistory()
  return session_store[session_id]

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

def make_chat_prompt():
    return ChatPromptTemplate.from_messages([
        (
            "system",
            """
Imagine you're a financial assistant.
Your job is to give grounded answers using only the context below.
Do not answer from outside the context.
Be friendly and simple, not robotic.

Context:
{sources}
""".strip()
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{query}")
    ])

In [17]:
# # See what's happening
# def _start(query):
#     return {"query": query}
from langchain_core.runnables import RunnablePassthrough

base_chain = (
    RunnablePassthrough()
    .assign(queries=RunnableLambda(lambda d: generate_multi_queries(d["query"], n = 5)))
    .assign(results=RunnableLambda(lambda d: retrieve_docs_multi(
        d["queries"],
        top_k_per_query=4, final_top_k=5
    )))
    .assign(sources=RunnableLambda(lambda d: build_context(d["results"], max_chars=1200)))
    .assign(prompt=RunnableLambda(lambda d: make_chat_prompt().invoke({
        "query": d["query"],
        "sources": d["sources"],
        "history": d.get("history", [])
    })))
    .assign(answer=RunnableLambda(lambda d: llm.invoke(d["prompt"])))
    .pick("answer")|StrOutputParser()
)

In [18]:
final_chain_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="query",
    history_messages_key="history",
)

In [19]:
response1 = final_chain_with_history.invoke(
    {"query": "What documents are needed for Loan Against Property?"},
    config={"configurable": {"session_id": "user_1"}}
)

In [20]:
print(response1)

Hi there! When you're looking into a Loan Against Property, you'll generally need a few key documents:

*   **Borrower KYC and income proof**: This is pretty standard, similar to what you'd need for a personal or home loan.
*   **Bank statements**: These help assess your cashflow and other financial commitments.
*   **Property documents**: You'll need your title deed, chain documents, tax receipts, the approved plan, and completion/occupancy documents.

Making sure these are all in order can help things go smoothly!


In [21]:
print(response1)

Hi there! When you're looking into a Loan Against Property, you'll generally need a few key documents:

*   **Borrower KYC and income proof**: This is pretty standard, similar to what you'd need for a personal or home loan.
*   **Bank statements**: These help assess your cashflow and other financial commitments.
*   **Property documents**: You'll need your title deed, chain documents, tax receipts, the approved plan, and completion/occupancy documents.

Making sure these are all in order can help things go smoothly!


In [22]:
response2 = final_chain_with_history.invoke(
    {"query": "Which loan did I ask for?"},
    config={"configurable": {"session_id": "user_1"}}
)

In [ ]:
print(response2)

Based on our previous conversation, you were asking about a **Loan Against Property (LAP)**.
